In [222]:
# Iván Andrés Trujillo Abella
# ivantrujillo1229@gmail.com

In [223]:
#!pip install thefuzz

In [224]:
import pandas as pd
import numpy as np
from thefuzz import fuzz
import re

In [225]:
def clean_text(text):
  text = text.strip().upper().replace(" ","")
  text = text.replace('Á','A')
  text = text.replace('É', 'E')
  text = text.replace('Í', 'I')
  text = text.replace('Ó', 'O')
  text = text.replace('Ú','U')
  return text

def drop_col(df_, threshold):
  df  = df_.copy()
  to_drop = []
  for col in df.columns:
    if (df[col].isnull().sum()/df.shape[0]) >= threshold:
      to_drop.append(col)
  return to_drop


def fix_cols(columns_name):
  new = []
  capture_year = ''
  for token in columns_name:
    if bool(len((re.findall('\d{4}', token)))):
      capture_year = ''.join(re.findall('\d{4}', token))
    new.append(str(capture_year) +  str(token) )
  return new

def capturing(string):
  if bool(re.search(r"(^\d{4}).+(Dólares|Pesos).+(FOB|CIF)", string)):
    result = re.search(r"(^\d{4}).+(Dólares|Pesos).+(FOB|CIF)", string)
    return '-'.join(result.groups())
  else:
    return string


def fuzzy_match(target, sample):
  book = {}
  book[target] = {}
  for element in sample:
    if fuzz.ratio(target, element) > 50 and len("".join(set(element) & set(target)))/len(min(element, target)) > 0.43:
      book[target][element] =  fuzz.ratio(target, element)
  array = sorted(book[target].items(), key=lambda item: int(item[1]), reverse=True)
  if len(array)>= 1:
    return sorted(book[target].items(), key=lambda item: int(item[1]), reverse=True)[0][0]
  else:
    return 'unknown'

In [226]:
url_names = 'https://raw.githubusercontent.com/it-ces/Datasets/main/world.csv'
names = pd.read_csv(url_names)
names.drop(index=names[names['es'].duplicated(keep='first')].index,inplace=True)
names['en'] = names['en'].apply(lambda x : clean_text(str(x)))
names['es'] = names['es'].apply(lambda x : clean_text(str(x)))
book = names[['en', 'es']].set_index('es').to_dict()['en']

In [227]:
url_loc = 'https://raw.githubusercontent.com/it-ces/Datasets/main/Countries-location(google).csv'
url_imp = 'https://raw.githubusercontent.com/it-ces/Datasets/main/dane-importaciones.csv'
url_exp = 'https://raw.githubusercontent.com/it-ces/Datasets/main/dane-exportaciones.csv'
url_pib = 'https://raw.githubusercontent.com/it-ces/Datasets/main/wbdata-gdp-lifeExp-time-series.csv'

In [228]:
info = {}
info['location']  = pd.read_csv(url_loc)
info['import'] = pd.read_csv(url_imp, header=[0,1])
info['export']  = pd.read_csv(url_exp, header=[0,1])
info['gdp'] = pd.read_csv(url_pib)
#info['url_names'] = pd.read_csv(url_names)

In [229]:
info['import'].drop(index=info['import'][info['import']['País']['Unnamed: 1_level_1'].duplicated(keep=False)].index, inplace=True)

In [230]:
info['export'][info['export']['País']['Unnamed: 1_level_1'].duplicated(keep=False)]

,Código,País,1970,Unnamed: 3_level_0,Unnamed: 4_level_0,Unnamed: 5_level_0,1971,Unnamed: 7_level_0,Unnamed: 8_level_0,Unnamed: 9_level_0,...,2022,Unnamed: 211_level_0,Unnamed: 212_level_0,Unnamed: 213_level_0,2023,Unnamed: 215_level_0,Unnamed: 216_level_0,2024*,Unnamed: 218_level_0,Unnamed: 219_level_0
,Unnamed: 0_level_1,Unnamed: 1_level_1,Kilos Netos,Dólares FOB,Pesos FOB,Unnamed: 5_level_1,Kilos Netos,Dólares FOB,Pesos FOB,Unnamed: 9_level_1,...,Kilos Netos,Dólares FOB,Pesos FOB,Unnamed: 213_level_1,Kilos Netos,Dólares FOB,Pesos FOB,Kilos Netos,Dólares FOB,Pesos FOB


In [231]:
info['export'].drop(columns= drop_col(info['export'], 0.95), inplace=True)
info['export'].columns = [col[0] + re.sub(r'Unnamed:.+', '',col[1]) for col in info['export'].columns]
info['export'].columns = fix_cols(info['export'].columns)
info['export'].columns = [capturing(name) for name in  info['export'].columns]
features_ = list(info['export'].columns[0:2]) + [ col for col in info['export'].columns if bool(len(re.findall(r'FOB',col)))]
info['export'] = info['export'][features_]

In [232]:
info['import'].drop(columns= drop_col(info['import'], 0.99), inplace=True)
info['import'].columns = [col[0] + re.sub(r'Unnamed:.+','',col[1]) for col in info['import'].columns]
info['import'].columns = fix_cols(info['import'].columns)
info['import'].columns = [capturing(name) for name in  info['import'].columns]
# capture FOBS
features_ = list(info['import'].columns[0:2]) + [ col for col in info['import'].columns if bool(len(re.findall(r'FOB',col)))]
info['import'] = info['import'][features_]

In [233]:
info

{'location':     country     latitude    longitude                  name
 0        AD   42.546.245    1.601.554               Andorra
 1        AE   23.424.076   53.847.818  United Arab Emirates
 2        AF    3.393.911   67.709.953           Afghanistan
 3        AG   17.060.816  -61.796.428   Antigua and Barbuda
 4        AI   18.220.554  -63.068.615              Anguilla
 ..      ...          ...          ...                   ...
 240      YE   15.552.727   48.516.388                 Yemen
 241      YT     -128.275   45.166.244               Mayotte
 242      ZA  -30.559.482   22.937.506          South Africa
 243      ZM  -13.133.897   27.849.332                Zambia
 244      ZW  -19.015.438   29.154.857              Zimbabwe
 
 [245 rows x 4 columns],
 'import':      Código                                               País  \
 1       NaN                                              Total   
 3      13.0                                         Afganistán   
 4      15.0      

In [234]:
keys = {}
keys['location']  = 'name'
keys['import'] = 'País'
keys['export'] = 'País'
keys['gdp']  =  'value'

In [235]:
for base in info:
  print(base)
  info[base].rename(columns={keys[base]:'id_key'}, inplace=True)
  info[base]['id_key'] = info[base]['id_key'].apply(lambda x : clean_text(str(x)))
  info[base]['id_key'] = info[base]['id_key'].replace(book)

location
import
export
gdp


In [236]:
all = []
for base in info:
  all = all + list(info[base]['id_key'].unique())
print(len(all))
print(len(set(all)))

gdpset = set(info['gdp']['id_key'].unique())
A = set(info['import']['id_key'].unique())
B = set(info['export']['id_key'].unique())
intersect = (A & B & gdpset)

to_keep = {}
for elemento in A - intersect:
  if fuzzy_match(elemento, gdpset-intersect) != 'unknown':
    if fuzzy_match(elemento, gdpset-intersect) not in to_keep.values():
      to_keep[elemento] =  fuzzy_match(elemento, gdpset-intersect)

1243
486


In [237]:
to_keep

{'CENTROAFRICANA,REPUBLICA': 'CENTRALAFRICANREPUBLIC',
 'EGYPT': 'EGYPT,ARABREP.',
 'HEARDYMCDONALD,ISLAS': 'CAYMANISLANDS',
 'COREA': 'KOREA,REP.',
 'ZFPEAGROSOLERA': 'FAROEISLANDS',
 'TURCASYCAICOS,ISLAS': 'TURKSANDCAICOSISLANDS',
 'TIMORDELESTE': 'TIMOR-LESTE',
 'GAMBIA': 'GAMBIA,THE',
 'TANZANIA,REPUBLICAUNIDADE': 'TANZANIA,UNITEDREPUBLICOF',
 'BAHAMAS': 'BAHAMAS,THE',
 'MICRONESIA,ESTADOSFEDERADOSDE': 'MICRONESIA,FED.STS.',
 'CEILAN': 'CHANNELISLANDS',
 'TOKELAU': 'BELARUS',
 "CÔTED'IVOIRE": "COTED'IVOIRE",
 'CONGO': 'CONGO,REP.',
 'COOK,ISLAS': 'SOLOMONISLANDS',
 'SAINTKITTSANDNEVIS': 'ST.KITTSANDNEVIS',
 'SLOVAKIA': 'SLOVAKREPUBLIC',
 'UNITEDSTATESOFAMERICA': 'UNITEDSTATES',
 'MACEDONIA': 'NORTHMACEDONIA',
 'MARSHALL,ISLAS': 'MARSHALLISLANDS',
 'SAHARAOCCIDENTAL': 'BAHRAIN',
 'KAZAJSTAN': 'KAZAKHSTAN',
 'NUEVAZELANDIA': 'NEWZEALAND',
 'CONGO,REPUBLICADEMOCRATICADEL': 'CONGO,DEM.REP.',
 'TÜRKIYE': 'TURKIYE',
 'IRAN,REPUBLICAISLAMICADE': 'IRAN,ISLAMICREP.',
 'CURAZAO,ISLA': 'CURAC

In [238]:
banned = ['ZFPESANVICENTEDEPAUL', 'ESCOCIA',
 'SWAZILANDIA',
'BOUVET,ISLA',
 'SVALBARDYJANMAYEN,ISLAS',
 'VENEZUELA,BOLIVARIANREPUBLICOF',
 'ANTILLASHOLANDESAS',
 'COOK,ISLAS',
 'SAHARAOCCIDENTAL',
 'MAN,ISLA',
 'MARTINIQUE',
 'CEILAN',
 'ANGLONORMANDAS,ISLAS',
  'MOLDAVIA,REPUBLICADE',
 'ALAND,ISLAS',
 'TOKELAU',
 'ZFPEPROTISA',
 'ZFPEAGROSOLERA',
 'ZFPSURCOLOMBIANA',
 'ALEMANIA,REPUBLICADEMOCRATICA',
 'MIDWAY,ISLAS',
  'SANMARTIN(PARTEHOLANDESA)',
 'HEARDYMCDONALD,ISLAS',
 'BONAIRE,ISLA',
 'NORFOLK,ISLAS', ]

new = {}
for unity in to_keep:
  if unity not in  banned:
    new[unity] = to_keep[unity]

In [239]:
new

{'CENTROAFRICANA,REPUBLICA': 'CENTRALAFRICANREPUBLIC',
 'EGYPT': 'EGYPT,ARABREP.',
 'COREA': 'KOREA,REP.',
 'TURCASYCAICOS,ISLAS': 'TURKSANDCAICOSISLANDS',
 'TIMORDELESTE': 'TIMOR-LESTE',
 'GAMBIA': 'GAMBIA,THE',
 'TANZANIA,REPUBLICAUNIDADE': 'TANZANIA,UNITEDREPUBLICOF',
 'BAHAMAS': 'BAHAMAS,THE',
 'MICRONESIA,ESTADOSFEDERADOSDE': 'MICRONESIA,FED.STS.',
 "CÔTED'IVOIRE": "COTED'IVOIRE",
 'CONGO': 'CONGO,REP.',
 'SAINTKITTSANDNEVIS': 'ST.KITTSANDNEVIS',
 'SLOVAKIA': 'SLOVAKREPUBLIC',
 'UNITEDSTATESOFAMERICA': 'UNITEDSTATES',
 'MACEDONIA': 'NORTHMACEDONIA',
 'MARSHALL,ISLAS': 'MARSHALLISLANDS',
 'KAZAJSTAN': 'KAZAKHSTAN',
 'NUEVAZELANDIA': 'NEWZEALAND',
 'CONGO,REPUBLICADEMOCRATICADEL': 'CONGO,DEM.REP.',
 'TÜRKIYE': 'TURKIYE',
 'IRAN,REPUBLICAISLAMICADE': 'IRAN,ISLAMICREP.',
 'CURAZAO,ISLA': 'CURACAO',
 'MACAO': 'MACAOSAR,CHINA',
 'HONGKONG': 'HONGKONGSAR,CHINA',
 'SIRIA,REPUBLICAARABE': 'SYRIANARABREPUBLIC',
 'RUSIA,FEDERACIONDE': 'RUSSIANFEDERATION',
 'YEMEN': 'YEMEN,REP.'}

In [240]:
for base in info:
  info[base]['id_key'] = info[base]['id_key'].replace(new)

In [241]:
len(gdpset & set(info['import']['id_key'].unique()) & set(info['export']['id_key'].unique()))

194

In [242]:
info['gdp_wide'] = info['gdp'].set_index(['id_key', 'year']).unstack(level=-1)

In [243]:
info['gdp_wide'].columns = [col[0] + str(col[1]) for col in info['gdp_wide'].columns]

In [244]:
info['gdp_wide'].reset_index(inplace=True)

In [245]:
# Fixing id_key's
book = {'COREA,REPUBLICADEMOCRATICA':"KOREA,DEM.PEOPLE'SREP.",
        'UNITEDKINGDOMOFGREATBRITAINANDNORTHERNIRELAND':'UNITEDKINGDOM',
        'VENEZUELA,BOLIVARIANREPUBLICOF':'VENEZUELA,RB',
        'VIRGENES(BRITANICAS),ISLAS':'BRITISHVIRGINISLANDS',
        'VIRGENES(DELOSESTADOSUNIDOS),ISLAS':'VIRGINISLANDS(U.S.)',
        'SAINTVINCENTANDTHEGRENADINES':'ST.VINCENTANDTHEGRENADINES',
        'SWAZILANDIA':'ESWATINI',
        'MAN,ISLA':'ISLEOFMAN',
        'LAOS,REPUBLICAPOPULARDEMOCRATICA':'LAOPDR',
        'CAIMAN,ISLAS':'CAYMANISLANDS',
        'CENTROAFRICANA,REPUBLICA':'CENTRALAFRICANREPUBLIC',
        'MARIANASDELNORTE,ISLAS':'NORTHERNMARIANAISLANDS',
        'SLOVAKIA':'SLOVAKREPUBLIC',
        'SANMARTIN(PARTEHOLANDESA)':'SINTMAARTEN(DUTCHPART)'}

info['export'].replace(book, inplace=True)
info['import'].replace(book, inplace=True)

In [246]:
full =info['gdp_wide'].merge(info['export'],
                       on='id_key',)

In [247]:
full.columns

Index(['id_key', 'incomeLevel1960', 'incomeLevel1961', 'incomeLevel1962',
       'incomeLevel1963', 'incomeLevel1964', 'incomeLevel1965',
       'incomeLevel1966', 'incomeLevel1967', 'incomeLevel1968',
       ...
       '2020-Dólares-FOB', '2020-Pesos-FOB', '2021-Dólares-FOB',
       '2021-Pesos-FOB', '2022-Dólares-FOB', '2022-Pesos-FOB',
       '2023-Dólares-FOB', '2023-Pesos-FOB', '2024-Dólares-FOB',
       '2024-Pesos-FOB'],
      dtype='object', length=360)

In [248]:
full = full.merge(info['import'], on='id_key', suffixes=('export','import'))

In [249]:
full.columns

Index(['id_key', 'incomeLevel1960', 'incomeLevel1961', 'incomeLevel1962',
       'incomeLevel1963', 'incomeLevel1964', 'incomeLevel1965',
       'incomeLevel1966', 'incomeLevel1967', 'incomeLevel1968',
       ...
       '2015-Dólares-FOBimport', '2016-Dólares-FOBimport',
       '2017-Dólares-FOBimport', '2018-Dólares-FOBimport',
       '2019-Dólares-FOBimport', '2020-Dólares-FOBimport',
       '2021-Dólares-FOBimport', '2022-Dólares-FOBimport',
       '2023-Dólares-FOBimport', '2024-Dólares-FOBimport'],
      dtype='object', length=386)

In [250]:
def format_coordinate(coor):
  coor = re.sub(r"\.", "_" ,coor, count=1)
  coor = re.sub(r"\.", "", coor)
  coor = re.sub(r'_', '.', coor)
  return float(coor)

In [251]:
info['location']['latitude'] = info['location']['latitude'].apply(lambda x: format_coordinate(str(x)))
info['location']['longitude'] = info['location']['longitude'].apply(lambda x: format_coordinate(str(x)))

In [252]:
def distance(longitudei, latitudei, longitudej, latitudej):
  return ((longitudei - longitudej)**2 + (latitudei - latitudej)**2)**0.5

In [253]:
col_latitude = 4.570868
col_longitude =  -74.297333

In [254]:
distance(col_longitude, col_latitude, col_longitude, col_latitude)

0.0

In [255]:
info['location']['distance'] = info['location'].apply(lambda col:  distance(col_longitude, col_latitude,  col['longitude'], col['latitude']), axis=1)

In [256]:
book = {'COREA,REPUBLICADEMOCRATICA':"KOREA,DEM.PEOPLE'SREP.",
        'UNITEDKINGDOMOFGREATBRITAINANDNORTHERNIRELAND':'UNITEDKINGDOM',
        'VENEZUELA,BOLIVARIANREPUBLICOF':'VENEZUELA,RB',
        'VIRGENES(BRITANICAS),ISLAS':'BRITISHVIRGINISLANDS',
        'VIRGENES(DELOSESTADOSUNIDOS),ISLAS':'VIRGINISLANDS(U.S.)',
        'SAINTVINCENTANDTHEGRENADINES':'ST.VINCENTANDTHEGRENADINES',
        'SWAZILANDIA':'ESWATINI',
        'MAN,ISLA':'ISLEOFMAN',
        'LAOS,REPUBLICAPOPULARDEMOCRATICA':'LAOPDR',
        'CAIMAN,ISLAS':'CAYMANISLANDS',
        'CENTROAFRICANA,REPUBLICA':'CENTRALAFRICANREPUBLIC',
        'MARIANASDELNORTE,ISLAS':'NORTHERNMARIANAISLANDS',
        'SLOVAKIA':'SLOVAKREPUBLIC',
        'SANMARTIN(PARTEHOLANDESA)':'SINTMAARTEN(DUTCHPART)'}

In [257]:
info['location'].replace(book, inplace=True)

In [258]:
full =full.merge(info['location'], on='id_key', how = 'outer', indicator=True)

In [259]:
full[full['_merge']=='left_only']['id_key'].unique()

array(['CABOVERDE', 'CONGO,DEM.REP.', 'CONGO,REP.', 'CURACAO', 'CZECHIA',
       'ESWATINI', 'IRAN,ISLAMICREP.', "KOREA,DEM.PEOPLE'SREP.",
       'KOREA,REP.', 'LAOPDR', 'MACAOSAR,CHINA', 'MICRONESIA,FED.STS.',
       'MYANMAR', 'NORTHMACEDONIA', 'RUSSIANFEDERATION',
       'SAOTOMEANDPRINCIPE', 'SINTMAARTEN(DUTCHPART)', 'SOUTHSUDAN',
       'SYRIANARABREPUBLIC', 'TURKIYE', 'VIRGINISLANDS(U.S.)'],
      dtype=object)

In [260]:
full[full['_merge']=='right_only']['id_key'].unique()

array(['ANGUILLA', 'ANTARCTICA', 'BAHRAIN', 'BELARUS', 'BOUVETISLAND',
       'BRITISHINDIANOCEANTERRITORY', 'CAPEVERDE', 'CHRISTMASISLAND',
       'COCOS[KEELING]ISLANDS', 'CONGO[DRC]', 'CONGO[REPUBLIC]',
       'COOKISLANDS', 'CZECHREPUBLIC', 'FALKLANDISLANDS[ISLASMALVINAS]',
       'FAROEISLANDS', 'FRENCHGUIANA', 'FRENCHSOUTHERNTERRITORIES',
       'GAZASTRIP', 'GUADELOUPE', 'GUERNSEY',
       'HEARDISLANDANDMCDONALDISLANDS', 'IRAN,ISLAMICREPUBLICOF',
       'JERSEY', 'KOSOVO', 'KYRGYZSTAN', "LAOPEOPLE'SDEMOCRATICREPUBLIC",
       'MACAU', 'MACEDONIA[FYROM]', 'MARTINIQUE', 'MAYOTTE',
       'MICRONESIA,FEDERATEDSTATESOF', 'MOLDOVA', 'MONTSERRAT',
       'MYANMAR[BURMA]', 'NETHERLANDSANTILLES', 'NIUE', 'NORFOLKISLAND',
       'NORTHKOREA', 'PALESTINIANTERRITORIES', 'PITCAIRNISLANDS',
       'REUNION', 'RUSSIA', 'SAINTHELENA', 'SAINTLUCIA',
       'SAINTPIERREANDMIQUELON', 'SOLOMONISLANDS',
       'SOUTHGEORGIAANDTHESOUTHSANDWICHISLANDS', 'SOUTHKOREA',
       'SVALBARDANDJANMAYEN', 'S

In [261]:
full['_merge'].value_counts()

,count
_merge,
both,185
right_only,60
left_only,21


In [262]:
full.columns

Index(['id_key', 'incomeLevel1960', 'incomeLevel1961', 'incomeLevel1962',
       'incomeLevel1963', 'incomeLevel1964', 'incomeLevel1965',
       'incomeLevel1966', 'incomeLevel1967', 'incomeLevel1968',
       ...
       '2020-Dólares-FOBimport', '2021-Dólares-FOBimport',
       '2022-Dólares-FOBimport', '2023-Dólares-FOBimport',
       '2024-Dólares-FOBimport', 'country', 'latitude', 'longitude',
       'distance', '_merge'],
      dtype='object', length=391)

In [264]:
test = full[['id_key', '2020-Dólares-FOBexport', '2020-Dólares-FOBimport', 'Gdp2020', 'distance']]

In [276]:
test

,id_key,2020-Dólares-FOBexport,2020-Dólares-FOBimport,Gdp2020,distance,exterior
0,AFGHANISTAN,"48,561","1,152",1968.341002,142.012163,"48,5611,152"
1,ALBANIA,"873,316","178,729",13278.434516,101.301719,"873,316178,729"
2,ALGERIA,"3,677,841","20,799,046",10844.770764,79.498257,"3,677,84120,799,046"
3,AMERICANSAMOA,0,"3,634",NaN,97.669545,"03,634"
4,ANDORRA,0,"167,186",NaN,84.869136,"0167,186"
...,...,...,...,...,...,...
261,WALLISANDFUTUNA,NaN,NaN,NaN,104.480941,NaN
262,WESTERNSAHARA,NaN,NaN,NaN,64.477010,NaN
263,"YEMEN,REP.","1,282,789",0,NaN,123.303736,"1,282,7890"
264,ZAMBIA,"120,842",0,3183.650773,103.669667,"120,8420"


In [268]:
test['exterior'] = test['2020-Dólares-FOBexport'] + test['2020-Dólares-FOBimport']

/tmp/ipython-input-268-3404487237.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['exterior'] = test['2020-Dólares-FOBexport'] + test['2020-Dólares-FOBimport']


In [275]:
test[['Gdp2020', 'distance', 'exterior']].dtypes

,0
Gdp2020,float64
distance,float64
exterior,object
